# Reto ingeniería de características

@author: Jessica Santizo Galicia.

In [76]:
# Descomentar la siguiente linea para instalar  la bibliotec scikit-learn en caso de que no se tenga
#!pip install scikit-learn

In [77]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.decomposition import PCA

2. Lee el archivo CSV llamado empleadosRETO.csv y coloca los datos en un frame de Pandas llamado EmpleadosAttrition.

In [78]:
EmpleadosAttrition = pd.read_csv('data/empleadosRETO.csv')

3. Elimina las columnas que, con alta probabilidad (estimada por ti), no tienen relación alguna con la salida. Hay algunas columnas que contienen información que no ayuda a definir el desgaste de un empleado, tal es caso de las siguientes:
EmployeeCount: número de empleados, todos tienen un 1
EmployeeNumber: ID del empleado, el cual es único para cada empleado
Over18: mayores de edad, todos dicen “Y”
StandardHours: horas de trabajo, todos tienen “80”

In [ ]:
EmpleadosAttrition = EmpleadosAttrition.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'])

4. Analiza la información proporcionada, si detectaste que no se cuenta con los años que el empelado lleva en la compañía y parece ser un buen dato. Dicha cantidad se puede calcular con la fecha de contratación ‘HiringDate’.

5. Se crea la columna Year con el año de contratación 

In [80]:

EmpleadosAttrition['Year'] = EmpleadosAttrition['HiringDate'].str.split('/').str[-1].astype(int)

6. Crea una columna llamada YearsAtCompany que contenga los años que el empleado lleva en la compañía hasta el año 2018. Para su cálculo, usa la variable Year que acabas de crear.

In [81]:
EmpleadosAttrition['Year'] = EmpleadosAttrition['HiringDate'].str.split('/').str[-1].astype(int)

7. La DistanceFromHome está dada en kilómetros, pero tiene las letras “km” al final y así no puede ser entera.

8. Renombra la variable DistanceFromHome a DistanceFromHome_km.

In [82]:
EmpleadosAttrition = EmpleadosAttrition.rename(columns={'DistanceFromHome': 'DistanceFromHome_km'})

 9. Crea una nueva variable DistanceFromHome que sea entera, es decir, solo con números.

In [83]:
EmpleadosAttrition['DistanceFromHome'] = EmpleadosAttrition['DistanceFromHome_km'].str.replace(' km', '', regex=False).astype(int)


10. Borra las columnas Year, HiringDate y DistanceFromHome_km debido a que ya no son útiles.

In [84]:
EmpleadosAttrition = EmpleadosAttrition.drop(columns=['Year', 'HiringDate', 'DistanceFromHome_km'])

11. Aprovechando los ajustes que se están haciendo, la empresa desea saber si todos los departamentos tienen un ingreso promedio similar. Genera una nuevo frame llamado SueldoPromedioDepto que contenga el MonthlyIncome promedio por departamento de los empleados y colócalo en una variable llamada SueldoPromedio. Esta tabla solo es informativa, no la vas a utilizar en el set de datos que estás construyendo.

In [85]:
SueldoPromedioDepto = EmpleadosAttrition.groupby('Department')['MonthlyIncome'].mean()
SueldoPromedioDepto = SueldoPromedioDepto.reset_index()
SueldoPromedioDepto = SueldoPromedioDepto.rename(columns={'MonthlyIncome': 'SueldoPromedio'})
print('***Sueldo promedio por departamento***')
print(SueldoPromedioDepto)

***Sueldo promedio por departamento***
               Department  SueldoPromedio
0         Human Resources     6239.888889
1  Research & Development     6804.149813
2                   Sales     7188.250000


12. La variable MonthlyIncome tiene un valor numérico muy grande comparada con las otras variables. Escala dicha variable para que tenga un valor entre 0 y 1. 

In [86]:
EmpleadosAttrition['MonthlyIncome'] = (
    (EmpleadosAttrition['MonthlyIncome'] - EmpleadosAttrition['MonthlyIncome'].min())
    / (EmpleadosAttrition['MonthlyIncome'].max() - EmpleadosAttrition['MonthlyIncome'].min())
)

In [87]:
#podemos usar minmaxscaler de Python y obtendremos el mismo resultado
#escalador = preprocessing.MinMaxScaler()
#EmpleadosAttrition['MonthlyIncome'] = escalador.fit_transform(EmpleadosAttrition[['MonthlyIncome']])

13. Todo parece indicar que las variables categóricas que quedan sí son importantes para obtener la variable de salida. Convierte todas las variables categóricas que quedan a numéricas:
- a) BusinessTravel
- b) Department
- c) EducationField
- d) Gender
- e) JobRole
- f) MaritalStatus
- g) Attrition

In [88]:
def convertir_a_numericas (df, columnas):
    """
    Convierte las columnas categóricas indicadas a valores numéricos
    usando LabelEncoder. 
    """
    for columna in columnas:
        # Se llenaron valores faltantes con la moda ya que LabelEncoder no acepta NaN
        if df[columna].isna().any():
            df[columna] = df[columna].fillna(df[columna].mode()[0])
        
        df[columna] = LabelEncoder().fit_transform(df[columna])
    
    return df


columnas_categoricas = [
    'BusinessTravel', 'Department', 'EducationField',
    'Gender', 'JobRole', 'MaritalStatus', 'Attrition'
]

EmpleadosAttrition = convertir_a_numericas(EmpleadosAttrition, columnas_categoricas)

14. Ahora debes hacer la evaluación de las variables para quedarte con las mejores. Calcula la correlación lineal de cada una de las variables con respecto al Attrition.

In [89]:

correlaciones = EmpleadosAttrition.corr(numeric_only=True)['Attrition']
print(correlaciones.sort_values(ascending=False))

Attrition                   1.000000
MaritalStatus               0.192430
JobRole                     0.078684
BusinessTravel              0.068650
Department                  0.054236
DistanceFromHome            0.052732
EducationField              0.051184
PerformanceRating          -0.006471
NumCompaniesWorked         -0.009082
WorkLifeBalance            -0.021723
Gender                     -0.028839
RelationshipSatisfaction   -0.030945
Education                  -0.055531
PercentSalaryHike          -0.060880
YearsSinceLastPromotion    -0.069000
TrainingTimesLastYear      -0.070884
EnvironmentSatisfaction    -0.124327
JobSatisfaction            -0.164957
JobInvolvement             -0.166785
MonthlyIncome              -0.194936
YearsInCurrentRole         -0.203918
Age                        -0.212121
TotalWorkingYears          -0.213329
JobLevel                   -0.214266
Name: Attrition, dtype: float64


15. Selecciona solo aquellas variables que tengan una correlación mayor o igual a 0.1, dejándolas en otro frame llamado EmpleadosAttritionFinal. No olvides mantener la variable de salidaAttrition; esto es equivalente a borrar las que no cumplen con el límite.

In [90]:
def seleccionar_variables(df, correlaciones, salida, umbral=0.1):
    """
    Selecciona las columnas de df cuya correlación (en valor absoluto)
    con la variable de salida es mayor o igual al umbral indicado.
    Siempre conserva la variable de salida.
    """
    variables_seleccionadas = correlaciones[correlaciones.abs() >= umbral].index.tolist()

    if salida not in variables_seleccionadas:
        variables_seleccionadas.append(salida)

    return df[variables_seleccionadas].copy()

EmpleadosAttritionFinal = seleccionar_variables(EmpleadosAttrition, correlaciones, salida='Attrition', umbral=0.1)

print(EmpleadosAttritionFinal.columns.tolist())
EmpleadosAttritionFinal.head()

['Age', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'TotalWorkingYears', 'YearsInCurrentRole', 'Attrition']


,Age,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MaritalStatus,MonthlyIncome,TotalWorkingYears,YearsInCurrentRole,Attrition
0,50,4,3,4,4,0,0.864269,32,4,0
1,36,2,3,2,2,0,0.207340,7,2,0
2,21,2,3,1,2,2,0.088062,1,0,1
3,52,2,3,3,2,2,0.497574,18,6,0
4,33,2,3,3,3,1,0.664470,15,6,1


16. Crea una nueva variable llamada EmpleadosAttritionPCA formada por los componentes principales del frame EmpleadosAttritionFinal. Recuerda que el resultado del proceso PCA es un numpy array, por lo que, para hacer referencia a una columna, por ejemplo, la 0, puedes usar la instrucción EmpleadosAttritionPCA[:,0]).


In [91]:
pca = PCA()
pca.fit(EmpleadosAttritionFinal.drop(columns=['Attrition']))

#no se toma en cuenta la variable dependiente
EmpleadosAttritionPCA = pca.transform(EmpleadosAttritionFinal.drop(columns=['Attrition']))

print(EmpleadosAttritionPCA[:1])

[[21.80432791  5.63351609 -5.80611807  0.2715007   2.11159229 -0.35407851
   0.67667271 -0.7622722   0.13336846]]


17. Agrega el mínimo número de Componentes Principales en columnas del frame EmpleadosAttritionPCA que logren explicar el 80% de la varianza, al frame EmpleadosAttritionFinal. Puedes usar la instrucción assign, columna por columna, llamando a cada unaC0, C1, etc., hasta las que vayas a agregar.

In [92]:
# 17. Determinar cuántos componentes de EmpleadosAttritionPCA explican el 80% de varianza
varianza_acumulada = np.cumsum(pca.explained_variance_ratio_)
n_componentes = np.argmax(varianza_acumulada >= 0.80) + 1

print('Número mínimo de componentes para explicar el 80% de la varianza ', n_componentes)
print('Varianza explicada acumulada', varianza_acumulada[:n_componentes])

# Agregando columnas C0, C1, ... usando assign
EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(
    **{f'C{i}': EmpleadosAttritionPCA[:, i] for i in range(n_componentes)}
)

# Print primeras 5 filas
EmpleadosAttritionFinal.head()

Número mínimo de componentes para explicar el 80% de la varianza  2
Varianza explicada acumulada [0.72926296 0.91074826]


,Age,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MaritalStatus,MonthlyIncome,TotalWorkingYears,YearsInCurrentRole,Attrition,C0,C1
0,50,4,3,4,4,0,0.864269,32,4,0,21.804328,5.633516
1,36,2,3,2,2,0,0.207340,7,2,0,-5.142815,-3.064859
2,21,2,3,1,2,2,0.088062,1,0,1,-20.647362,1.460521
3,52,2,3,3,2,2,0.497574,18,6,0,14.526880,-4.105516
4,33,2,3,3,3,1,0.664470,15,6,1,-1.773516,5.810337


18. Guarda el set de datos que has formado y que tienes en EmpleadosAttritionFinal en un archivo CSV llamado EmpleadosAttritionFinal.csv. Las últimas columnas que colocaste quedarán después de la variable Attrition, lo cual no importa, pero si gustas lo puedes arreglar antes de escribir el archivo.

In [93]:
columnas_ordenadas = [c for c in EmpleadosAttritionFinal.columns if c != 'Attrition'] + ['Attrition']
EmpleadosAttritionFinal = EmpleadosAttritionFinal[columnas_ordenadas]

EmpleadosAttritionFinal.to_csv('data/EmpleadosAttritionFinal.csv', index=False)
EmpleadosAttritionFinal.head()

,Age,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MaritalStatus,MonthlyIncome,TotalWorkingYears,YearsInCurrentRole,C0,C1,Attrition
0,50,4,3,4,4,0,0.864269,32,4,21.804328,5.633516,0
1,36,2,3,2,2,0,0.207340,7,2,-5.142815,-3.064859,0
2,21,2,3,1,2,2,0.088062,1,0,-20.647362,1.460521,1
3,52,2,3,3,2,2,0.497574,18,6,14.526880,-4.105516,0
4,33,2,3,3,3,1,0.664470,15,6,-1.773516,5.810337,1


19. Descarga tu script (archivo con extensión .ipynb) y guárdalo en un archivo que siga la nomenclatura que se te indica en Formato de entrega de actividad.  